# 04 — Assembling the summaries

**Purpose.** Combine the ten shard files `03` wrote into the single table the
next stage reads, and establish that the sample is covered — every hearing
summarized once, none twice, none missing.

**This notebook is where completeness is checked.** Each shard verifies its own
slice and knows nothing about the other nine, so a shard that died at the wall
clock leaves a file that passes its own checks while the sample has a hole. The
count against the source table here is what catches that, and it is an assert
rather than a printed number: a partial corpus that reads as complete is worse
than one that refuses to assemble.

**Inputs.** `interim/summaries/sst_{model}/shards/summaries_*.parquet` — from `03`.

**Output.** `data/cleaned/01_text_summarization/hearing_summaries.parquet`.

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, "../lib")

import pandas as pd

from paths import DATA_REPO, HEARINGS_REPO, summary_run_dir

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
RUN_NAME = os.environ.get("RUN_NAME", f"sst_{MODEL.split('/')[-1]}")
N_SHARDS = int(os.environ.get("N_SHARDS", 10))

RUN_DIR = summary_run_dir(RUN_NAME)
OUT_DIR = DATA_REPO / "cleaned" / "01_text_summarization"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = OUT_DIR / "hearing_summaries.parquet"

print(f"run    : {RUN_NAME}")
print(f"shards : {RUN_DIR / 'shards'}")
print(f"output : {TARGET}")

run    : sst_Qwen3.5-4B
shards : /scratch2/acroh/projects/llm_method/data/interim/summaries/sst_Qwen3.5-4B/shards
output : /project2/njia_1089/acroh/projects/llm_method/data/cleaned/01_text_summarization/hearing_summaries.parquet


## Step 1 — Read the shards

A missing shard file means a task never started or never wrote; that is reported
here rather than discovered as a gap in the row count later.

In [2]:
shard_files = sorted((RUN_DIR / "shards").glob("summaries_*.parquet"))
print(f"{len(shard_files)} of {N_SHARDS} shard files present")

missing_shards = set(range(N_SHARDS)) - {int(p.stem.split("_")[-1]) for p in shard_files}
if missing_shards:
    print(f"  MISSING SHARDS: {sorted(missing_shards)}")

frames = [pd.read_parquet(p) for p in shard_files]
summaries = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
for p, f in zip(shard_files, frames):
    ok = int((f["status"] == "ok").sum())
    print(f"  {p.name:<26} {len(f):>5,} rows, {ok:>5,} ok, "
          f"{f['seconds'].sum() / 3600:>5.2f} h")
print(f"\ntotal {len(summaries):,} rows")

10 of 10 shard files present
  summaries_00.parquet         236 rows,   236 ok, 10.64 h
  summaries_01.parquet         236 rows,   236 ok, 12.49 h
  summaries_02.parquet         235 rows,   235 ok, 12.91 h
  summaries_03.parquet         235 rows,   235 ok, 12.44 h
  summaries_04.parquet         235 rows,   235 ok, 10.95 h
  summaries_05.parquet         235 rows,   235 ok, 10.92 h
  summaries_06.parquet         235 rows,   235 ok,  9.99 h
  summaries_07.parquet         235 rows,   235 ok, 12.27 h
  summaries_08.parquet         235 rows,   235 ok, 10.07 h
  summaries_09.parquet         235 rows,   235 ok, 10.05 h

total 2,352 rows


## Step 2 — Join the hearing metadata

The summary alone is not usable downstream: topic extraction groups by committee
and by year, and relevance is scored against the hearing's own subject. Those
columns travel with the text rather than being re-joined by every later stage.

In [3]:
META = ["packageId", "title", "congress", "chamber", "year", "heldDate_first",
        "committee_ids", "committee_name", "committee_scope",
        "n_members", "n_witnesses", "n_words", "n_turns"]
sst = pd.read_parquet(HEARINGS_REPO / "hearings_sst.parquet", columns=META)

merged = sst.merge(
    summaries.drop(columns=["n_words", "n_turns"]),
    on="packageId", how="left", validate="one_to_one",
)
merged["summarized"] = merged["status"].eq("ok")

print(f"sample     {len(sst):>6,}")
print(f"summarized {int(merged['summarized'].sum()):>6,} "
      f"({merged['summarized'].mean():.1%})")
print(f"failed     {int(merged['status'].notna().sum() - merged['summarized'].sum()):>6,}")
print(f"never run  {int(merged['status'].isna().sum()):>6,}")

sample      2,352
summarized  2,352 (100.0%)
failed          0
never run       0


## Step 3 — Write the table

In [4]:
merged = merged.sort_values(["year", "packageId"]).reset_index(drop=True)
merged.to_parquet(TARGET, compression="zstd", index=False)

size = TARGET.stat().st_size
print(f"wrote {TARGET}")
print(f"  {len(merged):,} rows, {size / 1e6:.1f} MB zstd, {len(merged.columns)} columns")
good = merged.loc[merged["summarized"]]
print(f"  {good['summary'].str.split().str.len().sum():,} summary words "
      f"from {good['n_words'].sum() / 1e6:.1f}M transcript words "
      f"({good['summary'].str.split().str.len().sum() / good['n_words'].sum():.3%})")
print(f"  GPU time: {merged['seconds'].sum() / 3600:.1f} h")

wrote /project2/njia_1089/acroh/projects/llm_method/data/cleaned/01_text_summarization/hearing_summaries.parquet
  2,352 rows, 1.8 MB zstd, 24 columns
  638,687 summary words from 93.6M transcript words (0.682%)
  GPU time: 112.7 h


## Verification

In [5]:
back = pd.read_parquet(TARGET)

assert len(back) == len(sst), f"{len(back):,} rows against {len(sst):,} in the sample"
assert not back["packageId"].duplicated().any(), "a hearing appears twice"
print(f"1) one row per hearing, {len(back):,}, no duplicates")

# The shards must have partitioned the sample, not sampled it.
covered = back.loc[back["status"].notna()]
gap = len(back) - len(covered)
assert gap == 0, (
    f"{gap:,} hearings were never attempted — a shard did not finish. "
    f"Re-submit the array; 03 resumes from its logs."
)
print(f"2) every hearing was attempted by exactly one shard")

per_shard = covered.groupby("shard").size()
assert per_shard.notna().all() and len(per_shard) == N_SHARDS, (
    f"expected {N_SHARDS} shards, found {len(per_shard)}")
print(f"3) all {N_SHARDS} shards contributed "
      f"({per_shard.min():,}-{per_shard.max():,} hearings each)")

failed = back.loc[back["status"].notna() & ~back["summarized"]]
print(f"4) failures: {len(failed)}")
for r in failed.head(5).itertuples():
    print(f"     {r.packageId}: {r.status[:90]}")

good = back.loc[back["summarized"]]
assert (good["summary"].str.strip() != "").all(), "an ok row has no summary"
assert (good["summary_tokens"] <= 512).all(), "a summary exceeds the 512-token cap"
print(f"5) summaries: median {good['summary_tokens'].median():.0f} tokens, "
      f"max {good['summary_tokens'].max()}, all within the cap")

print("\n6) coverage by committee and decade:")
print(back.assign(decade=back["year"] // 10 * 10)
      .pivot_table(index="decade", columns="committee_scope",
                   values="summarized", aggfunc="mean")
      .map(lambda v: f"{v:.0%}" if pd.notna(v) else "-").to_string())

print("\n7) collapse rounds actually used:")
print(good["n_collapse_rounds"].value_counts().sort_index().to_string())

1) one row per hearing, 2,352, no duplicates
2) every hearing was attempted by exactly one shard
3) all 10 shards contributed (235-236 hearings each)
4) failures: 0
5) summaries: median 396 tokens, max 512, all within the cap

6) coverage by committee and decade:
committee_scope  both house_science senate_commerce
decade                                             
1990                -          100%            100%
2000             100%          100%            100%
2010                -          100%            100%
2020                -          100%            100%

7) collapse rounds actually used:
n_collapse_rounds
0    2259
1      93
